# SCS Indicator Manual Labeling Tool -- Multi-Chicken Edition (Google Colab)

Tool labeling 7 indikator SCS **per-ekor-ayam**. User cukup **upload satu file ZIP berisi foto-foto**, lalu labeli lewat widget.

## Cara pakai (3 langkah)
1. Menu **Runtime -> Run all** (Ctrl+F9).
2. Saat muncul tombol **Choose files**, pilih ZIP foto kamu (boleh banyak foto, boleh ada subfolder).
3. Scroll ke bawah ke widget labeling, labeli tiap ekor ayam, lalu klik **Export CSV** (file otomatis ter-download).

## Bagaimana "1 ekor ayam" ditentukan? (mode `MODE`)
| Isi ZIP | Mode | Perilaku |
|---|---|---|
| Foto polos (tanpa anotasi) | `detect` (default via `auto`) | YOLO mendeteksi kelas *bird* -> tiap box = 1 ekor. Foto tanpa deteksi dipakai utuh sebagai 1 unit. |
| Export Roboflow COCO (ada `_annotations.coco.json`) | `coco` (default via `auto`) | Sama seperti versi Kaggle: box badan+kepala yang tumpang tindih digabung jadi 1 ekor. |
| Foto polos, 1 foto = 1 ekor | `whole_image` | Tanpa deteksi, foto utuh = 1 unit. |

> Hasil deteksi otomatis itu **bantuan, bukan kebenaran**. Di widget ada gambar konteks (kotak merah = ayam yang sedang dinilai),
> jadi kalau crop-nya salah (dua ayam kegabung, atau bukan ayam) klik **Exclude** dan beri catatan.

## Catatan Colab
- Runtime Colab bersifat sementara. Progres tersimpan di `/content/scs_work` dan **hilang kalau runtime di-reset**.
  Untuk aman, set `SAVE_TO_DRIVE = True` di Konfigurasi, atau rajin klik **Export CSV** / download file progres.
- Kalau widget tidak muncul / tombol tidak merespons: jalankan ulang cell Setup, atau coba browser Chrome.

## 1. Setup

In [ ]:
#@title Setup -- jalankan sekali (±1 menit) { display-mode: "form" }
!pip install -q ultralytics

import os, io, re, csv, json, glob, shutil, zipfile, hashlib
from collections import defaultdict, Counter
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageOps, ImageDraw
from tqdm.auto import tqdm

# Supaya ipywidgets tampil benar di Colab (aman kalau gagal)
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception as e:
    print("Info: custom widget manager tidak diaktifkan:", e)

print("Setup selesai.")

## 2. Konfigurasi
Ubah lewat form di bawah (klik kanan -> *Show code* kalau mau lihat kodenya).

In [ ]:
#@title Konfigurasi { display-mode: "form" }

# --- Sumber foto ---
ZIP_SOURCE = "upload"  #@param ["upload", "drive"]
DRIVE_ZIP_PATH = "/content/drive/MyDrive/foto_ayam.zip"  #@param {type:"string"}
# (dipakai kalau ZIP_SOURCE = "drive"; boleh path file .zip ATAU folder berisi .zip)

# --- Penyimpanan hasil ---
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_WORK_DIR = "/content/drive/MyDrive/scs_labeling"  #@param {type:"string"}

# --- Cara menentukan "1 ekor ayam" ---
MODE = "auto"  #@param ["auto", "detect", "whole_image", "coco"]
DETECTOR_MODEL = "yolov8s.pt"  #@param ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]
DETECT_CONF = 0.25  #@param {type:"slider", min:0.05, max:0.9, step:0.05}
DETECT_IMGSZ = 960  #@param {type:"integer"}
MIN_BOX_AREA_RATIO = 0.01  #@param {type:"number"}
CROP_PADDING_RATIO = 0.08  #@param {type:"number"}
OVERLAP_THRESHOLD = 0.5  #@param {type:"number"}

# --- Tidak perlu diubah ---
DETECT_CLASSES = [14]   # 14 = "bird" di COCO (ayam paling sering terdeteksi sebagai bird)
SPLITS_NOTE = "split = nama folder relatif di dalam ZIP"
DATA_DIR = "/content/dataset"
UPLOAD_DIR = "/content/uploads"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

INDICATORS = [
    ("posisi_ekor", "Posisi ekor"),
    ("posisi_kepala", "Posisi kepala"),
    ("penutupan_mata", "Penutupan mata"),
    ("pembukaan_paruh", "Pembukaan paruh"),
    ("posisi_sayap", "Posisi sayap"),
    ("postur_kaki", "Postur kaki"),
    ("kondisi_bulu", "Kondisi bulu"),
]
IND_KEYS = [k for k, _ in INDICATORS]
LABEL_OPTIONS = ["not_stress", "stress", "undefined"]
STATE_OPTIONS = ["YA", "TIDAK", "TIDAK_TERLIHAT"]

if ZIP_SOURCE == "drive" or SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

WORK_DIR = DRIVE_WORK_DIR if SAVE_TO_DRIVE else "/content/scs_work"
os.makedirs(WORK_DIR, exist_ok=True)
PROGRESS_PATH = os.path.join(WORK_DIR, "indicators_progress_multi.json")
EXPORT_CSV_PATH = os.path.join(WORK_DIR, "indicators_ground_truth_multi.csv")
CACHE_PATH = os.path.join(WORK_DIR, "instances_cache.json")
print("Folder kerja :", WORK_DIR)
print("Mode         :", MODE)

## 3. Upload ZIP foto
Jalankan cell ini lalu pilih file `.zip` (atau langsung beberapa file foto). Folder bersarang di dalam ZIP tidak masalah.

In [ ]:
#@title Upload ZIP foto { display-mode: "form" }

def is_junk(path):
    parts = path.replace("\\", "/").split("/")
    return any(p == "__MACOSX" or p == ".DS_Store" or p.startswith("._") for p in parts)

def safe_extract_zip(zpath, dest):
    """Ekstrak zip dengan proteksi path-traversal & lewati file sampah macOS."""
    n = 0
    dest_real = os.path.realpath(dest)
    with zipfile.ZipFile(zpath) as zf:
        for m in zf.infolist():
            if m.is_dir() or is_junk(m.filename):
                continue
            target = os.path.realpath(os.path.join(dest, m.filename))
            if not target.startswith(dest_real + os.sep):
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with zf.open(m) as src, open(target, "wb") as out:
                shutil.copyfileobj(src, out)
            n += 1
    return n

shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

if ZIP_SOURCE == "upload":
    from google.colab import files
    shutil.rmtree(UPLOAD_DIR, ignore_errors=True)
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    os.chdir(UPLOAD_DIR)
    try:
        files.upload()
    finally:
        os.chdir("/content")
    sources = sorted(glob.glob(os.path.join(UPLOAD_DIR, "*")))
else:
    if os.path.isdir(DRIVE_ZIP_PATH):
        sources = sorted(glob.glob(os.path.join(DRIVE_ZIP_PATH, "*.zip")))
    else:
        sources = [DRIVE_ZIP_PATH]

n_files = 0
for s in sources:
    ext = os.path.splitext(s)[1].lower()
    if ext == ".zip":
        try:
            k = safe_extract_zip(s, DATA_DIR)
            print(f"[zip] {os.path.basename(s)} -> {k} file diekstrak")
            n_files += k
        except zipfile.BadZipFile:
            print(f"[!] {os.path.basename(s)} bukan ZIP yang valid -- dilewati")
    elif ext in IMG_EXT:
        shutil.copy(s, os.path.join(DATA_DIR, os.path.basename(s)))
        n_files += 1
    else:
        print(f"[!] Format tidak didukung, dilewati: {os.path.basename(s)}")

if n_files == 0:
    raise RuntimeError("Tidak ada file yang berhasil dimuat. Jalankan ulang cell ini dan pilih ZIP foto.")
print(f"\nTotal {n_files} file siap di {DATA_DIR}")

## 4. Tentukan instance per-ekor-ayam

- **coco**: box dengan kategori sama & tumpang tindih tinggi digabung jadi satu ekor (sama seperti versi Kaggle).
- **detect**: YOLO (COCO class *bird*) -> satu box = satu ekor, diurutkan kiri ke kanan. Foto tanpa deteksi dipakai utuh.
- **whole_image**: satu foto = satu unit.

In [ ]:
#@title Bangun daftar instance { display-mode: "form" }

def load_rgb(path):
    """Buka gambar + putar sesuai EXIF (penting untuk foto HP) + RGB."""
    im = Image.open(path)
    im = ImageOps.exif_transpose(im)
    return im.convert("RGB")

def find_images(root):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = sorted(d for d in dns if d != "__MACOSX" and not d.startswith("."))
        for fn in sorted(fns):
            if fn.startswith("._") or fn == ".DS_Store":
                continue
            if os.path.splitext(fn)[1].lower() in IMG_EXT:
                out.append(os.path.join(dp, fn))
    return out

def find_coco_jsons(root):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d != "__MACOSX"]
        if "_annotations.coco.json" in fns:
            out.append(os.path.join(dp, "_annotations.coco.json"))
    return sorted(out)

def rel_split(folder):
    r = os.path.relpath(folder, DATA_DIR).replace(os.sep, "/")
    return "root" if r == "." else r

# ---------- grouping untuk mode coco (identik dengan versi Kaggle) ----------
def box_xyxy(b):
    x, y, w, h = b
    return x, y, x + w, y + h

def overlap_ratio(small_box, big_box):
    sx1, sy1, sx2, sy2 = box_xyxy(small_box)
    bx1, by1, bx2, by2 = box_xyxy(big_box)
    iw = max(0, min(sx2, bx2) - max(sx1, bx1))
    ih = max(0, min(sy2, by2) - max(sy1, by1))
    small_area = max(1e-6, (sx2 - sx1) * (sy2 - sy1))
    return (iw * ih) / small_area

def group_instances(anns):
    anns_sorted = sorted(anns, key=lambda a: a["bbox"][2] * a["bbox"][3], reverse=True)
    used = [False] * len(anns_sorted)
    groups = []
    for i, a in enumerate(anns_sorted):
        if used[i]:
            continue
        used[i] = True
        group = [a]
        for j in range(i + 1, len(anns_sorted)):
            if used[j]:
                continue
            b = anns_sorted[j]
            if b["category_id"] == a["category_id"] and overlap_ratio(b["bbox"], a["bbox"]) > OVERLAP_THRESHOLD:
                used[j] = True
                group.append(b)
        groups.append(group)
    return groups

def make_inst(split, file_name, image_path, W, H, box, ref, n_boxes, gi, source):
    return {
        "split": split, "file_name": file_name, "image_path": image_path,
        "img_w": W, "img_h": H, "crop_box": [float(v) for v in box],
        "ref_category": ref, "n_boxes_in_group": n_boxes,
        "instance_key": f"{split}/{file_name}#inst{gi}", "inst_idx": gi, "source": source,
    }

def build_coco(coco_files):
    out, skipped_missing = [], 0
    for ann_path in coco_files:
        folder = os.path.dirname(ann_path)
        split = rel_split(folder)
        d = json.load(open(ann_path))
        cats = {c["id"]: c["name"] for c in d["categories"]}
        imgs = {im["id"]: im for im in d["images"]}
        by_img = defaultdict(list)
        for a in d["annotations"]:
            by_img[a["image_id"]].append(a)
        for img_id, anns in by_img.items():
            info = imgs[img_id]
            ipath = os.path.join(folder, info["file_name"])
            if not os.path.exists(ipath):
                skipped_missing += 1
                continue
            for gi, group in enumerate(group_instances(anns)):
                rep = max(group, key=lambda a: a["bbox"][2] * a["bbox"][3])
                out.append(make_inst(split, info["file_name"], ipath, info["width"], info["height"],
                                     rep["bbox"], cats[rep["category_id"]], len(group), gi, "coco"))
        n_no_ann = len(set(imgs) - set(by_img))
        if n_no_ann:
            print(f"   [info] {split}: {n_no_ann} gambar tanpa anotasi dilewati (mode coco)")
    if skipped_missing:
        print(f"   [!] {skipped_missing} gambar ada di JSON tapi file-nya tidak ada di ZIP -- dilewati")
    return out

def build_whole(images):
    out = []
    for p in tqdm(images, desc="whole_image"):
        W, H = load_rgb(p).size
        out.append(make_inst(rel_split(os.path.dirname(p)), os.path.basename(p), p, W, H,
                             (0, 0, W, H), "foto_utuh", 1, 0, "whole_image"))
    return out

def build_detect(images):
    from ultralytics import YOLO
    model = YOLO(DETECTOR_MODEL)
    out, n_fallback = [], 0
    for p in tqdm(images, desc="deteksi ayam"):
        im = load_rgb(p)
        W, H = im.size
        arr = np.ascontiguousarray(np.asarray(im)[:, :, ::-1])   # RGB -> BGR (konvensi ultralytics untuk numpy)
        res = model.predict(arr, conf=DETECT_CONF, classes=DETECT_CLASSES,
                            imgsz=DETECT_IMGSZ, verbose=False)[0]
        dets = []
        for b in res.boxes:
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].tolist()]
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
            w, h = x2 - x1, y2 - y1
            if w <= 1 or h <= 1 or (w * h) < MIN_BOX_AREA_RATIO * W * H:
                continue
            dets.append((x1, y1, w, h, float(b.conf[0])))
        dets.sort(key=lambda t: (t[0], t[1]))    # kiri -> kanan
        split, fname = rel_split(os.path.dirname(p)), os.path.basename(p)
        if not dets:
            n_fallback += 1
            out.append(make_inst(split, fname, p, W, H, (0, 0, W, H),
                                 "tidak_ada_deteksi(foto_utuh)", 1, 0, "detect_fallback"))
        else:
            for gi, (x, y, w, h, c) in enumerate(dets):
                out.append(make_inst(split, fname, p, W, H, (x, y, w, h),
                                     f"terdeteksi_bird(conf={c:.2f})", 1, gi, "detect"))
    print(f"   Foto tanpa deteksi (dipakai utuh): {n_fallback} dari {len(images)}")
    return out

# ---------- pilih mode ----------
all_images = find_images(DATA_DIR)
coco_files = find_coco_jsons(DATA_DIR)
mode = MODE
if mode == "auto":
    mode = "coco" if coco_files else "detect"
if mode == "coco" and not coco_files:
    print("[!] Mode coco dipilih tapi _annotations.coco.json tidak ditemukan -> pakai mode detect")
    mode = "detect"
print(f"Ditemukan {len(all_images)} gambar, {len(coco_files)} file anotasi COCO -> mode efektif: {mode}")
if not all_images:
    raise RuntimeError("Tidak ada gambar (jpg/png/...) di dalam ZIP.")

# ---------- cache: hasil deteksi tidak dihitung ulang kalau input & parameter sama ----------
sig_src = json.dumps([
    mode, DETECTOR_MODEL if mode == "detect" else "", DETECT_CONF if mode == "detect" else 0,
    DETECT_IMGSZ if mode == "detect" else 0, MIN_BOX_AREA_RATIO if mode == "detect" else 0,
    OVERLAP_THRESHOLD if mode == "coco" else 0,
    [(os.path.relpath(p, DATA_DIR), os.path.getsize(p)) for p in all_images],
    [(os.path.relpath(c, DATA_DIR), os.path.getsize(c)) for c in coco_files] if mode == "coco" else [],
])
sig = hashlib.md5(sig_src.encode()).hexdigest()

instances = None
if os.path.exists(CACHE_PATH):
    try:
        cache = json.load(open(CACHE_PATH))
        if cache.get("sig") == sig:
            instances = cache["instances"]
            print(f"Instance dimuat dari cache ({len(instances)} ekor).")
    except Exception:
        instances = None

if instances is None:
    if mode == "coco":
        instances = build_coco(coco_files)
    elif mode == "whole_image":
        instances = build_whole(all_images)
    else:
        instances = build_detect(all_images)
    json.dump({"sig": sig, "instances": instances}, open(CACHE_PATH, "w"))

instances.sort(key=lambda r: (r["split"], r["file_name"], r["inst_idx"]))
if not instances:
    raise RuntimeError("Tidak ada instance yang terbentuk. Coba MODE = 'whole_image' atau turunkan DETECT_CONF.")

per_file = Counter((r["split"], r["file_name"]) for r in instances)
print(f"\nTotal instance (unit yang akan dilabeli) : {len(instances)}")
print(f"Gambar dengan >1 ekor ayam               : {sum(1 for v in per_file.values() if v > 1)} file")

# peta file -> box semua instance (untuk gambar konteks)
boxes_by_file = defaultdict(list)
for r in instances:
    boxes_by_file[r["image_path"]].append(r["crop_box"])

## 5. Muat progres sebelumnya (opsional)
Kalau kamu pernah berhenti di tengah jalan dan punya file `indicators_progress_multi.json`, set `UPLOAD_PROGRESS = True` lalu jalankan cell ini. Kalau `SAVE_TO_DRIVE = True`, progres di Drive otomatis dimuat.

In [ ]:
#@title Muat / siapkan progres { display-mode: "form" }
UPLOAD_PROGRESS = False  #@param {type:"boolean"}

def blank_entry():
    return {"indicators": {k: None for k in IND_KEYS}, "label": "", "notes": "", "excluded": False}

if UPLOAD_PROGRESS:
    from google.colab import files
    os.makedirs("/content/progress_in", exist_ok=True)
    os.chdir("/content/progress_in")
    try:
        up = files.upload()
    finally:
        os.chdir("/content")
    for name, data in up.items():
        if name.endswith(".json"):
            with open(PROGRESS_PATH, "wb") as f:
                f.write(data)
            print("Progres di-upload:", name)
            break

if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH) as f:
        progress = json.load(f)
    print(f"Progres sebelumnya dimuat: {len(progress)} entri.")
else:
    progress = {}
    print("Belum ada progres -- mulai dari nol.")

def get_entry(key):
    if key not in progress:
        progress[key] = blank_entry()
    e = progress[key]
    for k in IND_KEYS:                       # jaga-jaga kalau skema progres lama kurang kunci
        e["indicators"].setdefault(k, None)
    return e

def save_progress():
    tmp = PROGRESS_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(progress, f, indent=2)
    os.replace(tmp, PROGRESS_PATH)

def is_complete(e):
    return all(e["indicators"].get(k) is not None for k in IND_KEYS) and bool(e["label"])

## 6. Widget labeling per-ekor-ayam

- **Kiri**: crop ayam yang dinilai. **Kanan**: foto utuh, kotak **merah** = ayam yang sedang dinilai, kotak abu = ayam lain di foto yang sama.
- Indikator yang belum disentuh tampil **tanpa pilihan** (tidak lagi otomatis dianggap "TIDAK"). Instance dihitung "lengkap" kalau 7 indikator + label akhir sudah diisi.
- Setiap klik tersimpan otomatis.

In [ ]:
state = {"idx": 0}
_suppress = {"on": False}

crop_output = widgets.Output(layout=widgets.Layout(min_width="300px"))
ctx_output = widgets.Output(layout=widgets.Layout(min_width="300px"))
fname_label = widgets.Label()
ref_label = widgets.Label()
progress_label = widgets.Label()
suggest_label = widgets.Label()

DW = {"description_width": "150px"}
indicator_widgets = {
    k: widgets.ToggleButtons(options=STATE_OPTIONS, value=None, description=lbl, style=DW)
    for k, lbl in INDICATORS
}
label_dropdown = widgets.Dropdown(options=LABEL_OPTIONS, value=None, description="Label akhir:", style=DW)
notes_box = widgets.Text(description="Catatan:", placeholder="opsional", style=DW, layout=widgets.Layout(width="500px"))

prev_btn = widgets.Button(description="<- Sebelumnya")
next_btn = widgets.Button(description="Berikutnya ->")
next_todo_btn = widgets.Button(description="Lompat ke belum lengkap", button_style="info")
exclude_btn = widgets.Button(description="Exclude instance ini", button_style="danger")
reset_btn = widgets.Button(description="Kosongkan isian")
export_btn = widgets.Button(description="Export CSV + Download", button_style="success")
export_output = widgets.Output()

def get_views(inst):
    im = load_rgb(inst["image_path"])
    W, H = im.size
    x, y, w, h = inst["crop_box"]
    px, py = w * CROP_PADDING_RATIO, h * CROP_PADDING_RATIO
    crop = im.crop((int(max(0, x - px)), int(max(0, y - py)), int(min(W, x + w + px)), int(min(H, y + h + py))))
    crop.thumbnail((450, 450))

    ctx = im.copy()
    d = ImageDraw.Draw(ctx)
    lw = max(3, W // 250)
    for b in boxes_by_file.get(inst["image_path"], []):
        if [float(v) for v in b] != [float(v) for v in inst["crop_box"]]:
            d.rectangle([b[0], b[1], b[0] + b[2], b[1] + b[3]], outline=(150, 150, 150), width=lw)
    d.rectangle([x, y, x + w, y + h], outline=(255, 0, 0), width=lw + 2)
    ctx.thumbnail((450, 450))
    return crop, ctx

def update_summary(entry, saved=False):
    ya = sum(1 for k in IND_KEYS if entry["indicators"][k] == "YA")
    n_set = sum(1 for k in IND_KEYS if entry["indicators"][k] is not None)
    txt = f"Terisi {n_set}/7 indikator | Total YA: {ya}/7"
    if is_complete(entry):
        txt += " | LENGKAP"
    suggest_label.value = txt + (" (tersimpan)" if saved else "")

def update_progress_label():
    n_done = sum(1 for inst in instances if is_complete(get_entry(inst["instance_key"])))
    entry = get_entry(instances[state["idx"]]["instance_key"])
    progress_label.value = (f"Instance {state['idx']+1} / {len(instances)}   |   lengkap: {n_done}/{len(instances)}"
                            + ("   [DIKECUALIKAN]" if entry["excluded"] else ""))

def render_current():
    _suppress["on"] = True
    try:
        idx = state["idx"]
        inst = instances[idx]
        entry = get_entry(inst["instance_key"])
        crop, ctx = get_views(inst)
        with crop_output:
            clear_output(wait=True); display(crop)
        with ctx_output:
            clear_output(wait=True); display(ctx)
        fname_label.value = f"{inst['split']}/{inst['file_name']}  (ekor ke-{inst['inst_idx']+1} di foto ini)"
        ref_label.value = f"Referensi (bukan ground truth): {inst['ref_category']} | {inst['n_boxes_in_group']} box digabung"
        for k in IND_KEYS:
            indicator_widgets[k].value = entry["indicators"][k]
        label_dropdown.value = entry["label"] or None
        notes_box.value = entry["notes"] or ""
        update_summary(entry)
        update_progress_label()
        prev_btn.disabled = idx == 0
        next_btn.disabled = idx == len(instances) - 1
    finally:
        _suppress["on"] = False

def _current_entry():
    return get_entry(instances[state["idx"]]["instance_key"])

def make_ind_handler(k):
    def handler(change):
        if _suppress["on"]:
            return
        e = _current_entry()
        e["indicators"][k] = change["new"]
        save_progress(); update_summary(e, saved=True); update_progress_label()
    return handler

def on_label(change):
    if _suppress["on"]:
        return
    e = _current_entry()
    e["label"] = change["new"] or ""
    save_progress(); update_summary(e, saved=True); update_progress_label()

def on_notes(change):
    if _suppress["on"]:
        return
    e = _current_entry()
    e["notes"] = change["new"]
    save_progress()

for k, w in indicator_widgets.items():
    w.observe(make_ind_handler(k), names="value")
label_dropdown.observe(on_label, names="value")
notes_box.observe(on_notes, names="value")

def go(delta):
    new = state["idx"] + delta
    if 0 <= new < len(instances):
        state["idx"] = new
        render_current()

def on_next_todo(_):
    n = len(instances)
    for step in range(1, n + 1):
        i = (state["idx"] + step) % n
        e = get_entry(instances[i]["instance_key"])
        if not is_complete(e) and not e["excluded"]:
            state["idx"] = i
            render_current()
            return
    with export_output:
        clear_output(wait=True)
        print("Semua instance sudah lengkap atau dikecualikan.")

def on_exclude(_):
    e = _current_entry()
    e["excluded"] = not e["excluded"]
    save_progress(); render_current()

def on_reset(_):
    progress[instances[state["idx"]]["instance_key"]] = blank_entry()
    save_progress(); render_current()

def try_download(path):
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        print("Download otomatis tidak tersedia. Ambil dari panel Files (ikon folder di kiri):", path)

def do_export(download=True):
    with open(EXPORT_CSV_PATH, "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["instance_key", "split", "file_name", "ref_category_roboflow"] + IND_KEYS +
                    ["label_final", "notes", "excluded",
                     "source_mode", "crop_x", "crop_y", "crop_w", "crop_h", "img_w", "img_h"])
        for inst in instances:
            e = get_entry(inst["instance_key"])
            x, y, w, h = inst["crop_box"]
            wr.writerow([inst["instance_key"], inst["split"], inst["file_name"], inst["ref_category"]] +
                        [e["indicators"][k] or "" for k in IND_KEYS] +
                        [e["label"], e["notes"], int(e["excluded"]),
                         inst["source"], round(x, 1), round(y, 1), round(w, 1), round(h, 1), inst["img_w"], inst["img_h"]])
    save_progress()
    n_done = sum(1 for inst in instances if is_complete(get_entry(inst["instance_key"])))
    n_ex = sum(1 for inst in instances if get_entry(inst["instance_key"])["excluded"])
    print(f"Tersimpan ke: {EXPORT_CSV_PATH}")
    print(f"{n_done}/{len(instances)} instance lengkap (7 indikator + label akhir), {n_ex} dikecualikan.")
    if download:
        try_download(EXPORT_CSV_PATH)
        try_download(PROGRESS_PATH)

def on_export(_):
    with export_output:
        clear_output(wait=True)
        do_export(download=True)

prev_btn.on_click(lambda _: go(-1))
next_btn.on_click(lambda _: go(+1))
next_todo_btn.on_click(on_next_todo)
exclude_btn.on_click(on_exclude)
reset_btn.on_click(on_reset)
export_btn.on_click(on_export)

ui = widgets.VBox([
    progress_label,
    widgets.HBox([crop_output, ctx_output]),
    fname_label,
    ref_label,
    widgets.VBox(list(indicator_widgets.values())),
    label_dropdown,
    suggest_label,
    notes_box,
    widgets.HBox([prev_btn, next_btn, next_todo_btn]),
    widgets.HBox([exclude_btn, reset_btn]),
    widgets.HBox([export_btn]),
    export_output,
])

display(ui)
render_current()

## 7. Export / download (cadangan)
Kalau tombol **Export CSV + Download** di widget tidak men-download file, jalankan cell ini. Hasil juga ada di panel **Files** (ikon folder di kiri) pada folder `scs_work`.

In [ ]:
#@title Export & download hasil { display-mode: "form" }
do_export(download=True)

## 8. Catatan penting

- `instance_key` formatnya `split/file_name#instN`. `split` = nama folder relatif di dalam ZIP (`root` kalau foto ada di level paling atas).
  Satu foto bisa muncul beberapa kali di CSV kalau ada beberapa ekor ayam. Kunci ini yang dipakai untuk menggabung ke pipeline.
- Kolom `ref_category_roboflow` hanya referensi (kategori Roboflow di mode `coco`, atau info deteksi YOLO di mode `detect`),
  **bukan** ground truth. `label_final` dan 7 indikator murni dari penilaian manual.
- Kolom tambahan di CSV: `source_mode` (`coco` / `detect` / `detect_fallback` / `whole_image`), `crop_x/y/w/h` (box di foto yang sudah di-rotasi sesuai EXIF), `img_w/h`.
- Deteksi otomatis (mode `detect`) memakai kelas *bird* dari model COCO. Bisa meleset: ayam bertumpuk/tertutup, ayam kecil di kejauhan,
  atau satu kotak berisi dua ayam. Cek gambar konteks (kotak merah); kalau crop tidak mewakili satu ekor dengan bersih,
  **Exclude** dan isi catatan. Kalau terlalu banyak yang terlewat, turunkan `DETECT_CONF` atau naikkan `DETECT_IMGSZ`; kalau banyak false positive, naikkan `MIN_BOX_AREA_RATIO`.
- Kalau melanjutkan sesi lama, pakai **parameter yang sama** (terutama `MODE`, `DETECT_CONF`) dan ZIP yang sama supaya `instance_key` tetap cocok dengan progres.
- Progres otomatis tersimpan setiap klik di `WORK_DIR`. Di Colab tanpa Drive, folder itu hilang saat runtime di-reset -- download CSV/progres sebelum menutup.